# Module 4 — Search evaluation (lessons 01–06)

Course: [04-evaluation](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/04-evaluation)

Until now, "search works" meant eyeballing a few queries. Evaluation replaces that with a number:
a **ground truth** of questions whose correct document is known, a search function, and metrics.

- **Hit Rate@5**: share of questions whose correct document appears anywhere in the top 5.
- **MRR@5**: mean of `1/rank` of the correct document (0 if missing). Rank 1 → 1.0, rank 5 → 0.2.

Ground truth: `data/ground_truth-faq.csv`, LLM-generated by the course (lessons 02–03), 5 questions per
LLM Zoomcamp FAQ record. Each question is labelled with the `document` id it came from. Generating it again
would cost ~80 calls of the free-tier budget, so the prepared file is used here. Homework 4 generates a sample itself.

In [1]:
import pandas as pd

from eval_utils import evaluate, hit_rate, mrr
from ingest import build_index, load_faq_data

ground_truth = pd.read_csv("data/ground_truth-faq.csv").to_dict(orient="records")

documents = [d for d in load_faq_data() if d["course"] == "llm-zoomcamp"]
index = build_index(documents)

print(len(ground_truth), "questions over", len(documents), "FAQ records")
ground_truth[0]

395 questions over 161 FAQ records


{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

## Relevance lists (lesson 04)

For each question, run search and mark each result 1 if it is the labelled document, else 0.
`[0, 0, 1, 0, 0]` means the right record came third.

In [2]:
def text_search(query, boost=None, num_results=5):
    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost or {"question": 3.0, "section": 0.5},
    )


def is_relevant(q, doc):
    return doc["id"] == q["document"]


for q in ground_truth[:3]:
    print([int(is_relevant(q, d)) for d in text_search(q["question"])], "-", q["question"])

[1, 0, 0, 0, 0] - Is it okay to join the course late if I just found it now?
[0, 0, 1, 0, 0] - Can I still take this course even if I missed the start date?
[1, 0, 0, 0, 0] - If I join after the course has already started, am I still eligible for a certificate?


## Metrics (lesson 05)

`hit_rate`, `mrr` and `evaluate` live in [`eval_utils.py`](eval_utils.py). `evaluate` takes the relevance rule as a
parameter, so the homework can reuse it with lesson filenames instead of FAQ ids.

In [3]:
print("hit rate of [[1,0],[0,0],[0,1]]:", hit_rate([[1, 0], [0, 0], [0, 1]]))
print("mrr      of [[1,0],[0,0],[0,1]]:", mrr([[1, 0], [0, 0], [0, 1]]))

baseline = evaluate(ground_truth, text_search, is_relevant)
baseline

hit rate of [[1,0],[0,0],[0,1]]: 0.6666666666666666
mrr      of [[1,0],[0,0],[0,1]]: 0.5


  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.620253164556962, 'mrr': 0.5427848101265824}

## Tuning boosts (lesson 06)

Module 1 picked `question: 3.0, section: 0.5` by intuition. With a fixed ground truth, every choice can be measured.
Grid over three field boosts:

In [4]:
import itertools

rows = []
for qb, ab, sb in itertools.product([0.5, 1.0, 2.0, 3.0], [0.5, 1.0, 2.0], [0.0, 0.1, 0.5]):
    boost = {"question": qb, "answer": ab, "section": sb}
    metrics = evaluate(ground_truth, lambda query: text_search(query, boost), is_relevant, progress=False)
    rows.append({**boost, **metrics})

df_boost = pd.DataFrame(rows).sort_values("mrr", ascending=False)
df_boost.head(8)

,question,answer,section,hit_rate,mrr
3,0.5,1.0,0.0,0.724051,0.628608
15,1.0,2.0,0.0,0.724051,0.628608
16,1.0,2.0,0.1,0.724051,0.626624
4,0.5,1.0,0.1,0.721519,0.624473
7,0.5,2.0,0.1,0.726582,0.613840
6,0.5,2.0,0.0,0.726582,0.613629
17,1.0,2.0,0.5,0.708861,0.609536
0,0.5,0.5,0.0,0.696203,0.605992


In [5]:
best = df_boost.iloc[0]
print(f"baseline (question=3, section=0.5): hit_rate={baseline['hit_rate']:.3f}  mrr={baseline['mrr']:.3f}")
print(f"best grid point:                    hit_rate={best['hit_rate']:.3f}  mrr={best['mrr']:.3f}")
print(df_boost.groupby("question")["mrr"].mean().round(3).to_dict(), "<- mean MRR by question boost")
print(df_boost.groupby("answer")["mrr"].mean().round(3).to_dict(), "<- mean MRR by answer boost")

baseline (question=3, section=0.5): hit_rate=0.620  mrr=0.543
best grid point:                    hit_rate=0.724  mrr=0.629
{0.5: 0.598, 1.0: 0.587, 2.0: 0.562, 3.0: 0.542} <- mean MRR by question boost
{0.5: 0.541, 1.0: 0.575, 2.0: 0.601} <- mean MRR by answer boost


### Caveat: the tuned number is optimistic

The boosts were chosen *on* the same 395 questions they are scored on, so the best grid point overfits a little.
The honest procedure is to tune on one split and report on another:

In [6]:
import random

random.seed(1)
ids = sorted({q["document"] for q in ground_truth})
random.shuffle(ids)
val_ids = set(ids[: len(ids) // 2])   # split by document so questions about one record don't leak across
val = [q for q in ground_truth if q["document"] in val_ids]
test = [q for q in ground_truth if q["document"] not in val_ids]

scores = []
for qb, ab, sb in itertools.product([0.5, 1.0, 2.0, 3.0], [0.5, 1.0, 2.0], [0.0, 0.1, 0.5]):
    boost = {"question": qb, "answer": ab, "section": sb}
    scores.append((evaluate(val, lambda query: text_search(query, boost), is_relevant, progress=False)["mrr"], boost))
best_val_mrr, best_boost = max(scores, key=lambda s: s[0])

test_tuned = evaluate(test, lambda query: text_search(query, best_boost), is_relevant, progress=False)
test_base = evaluate(test, text_search, is_relevant, progress=False)
print("boost chosen on validation:", best_boost, f"(val mrr {best_val_mrr:.3f})")
print(f"test mrr  baseline={test_base['mrr']:.3f}  tuned={test_tuned['mrr']:.3f}")

boost chosen on validation: {'question': 0.5, 'answer': 1.0, 'section': 0.0} (val mrr 0.646)
test mrr  baseline=0.520  tuned=0.612


**Findings**

- The module 1 boost (`question: 3.0`, chosen by intuition) is the **worst** question-boost setting in the grid.
  Mean MRR falls as the question boost rises (0.60 → 0.54) and rises with the answer boost (0.54 → 0.60).
- Why: the ground-truth questions were generated *from the whole FAQ record* and deliberately reworded,
  so they share more vocabulary with answers than with the short FAQ question titles.
  User questions behave the same way. The boost that "felt right" was tuned to the wrong field.
- The gain holds on held-out documents: tuned on one half, test MRR **0.520 → 0.612** (+18%).
  The full-data best (0.629) is a little optimistic, as expected.
- Caveat: the ground truth is LLM-generated, so it measures how well search finds *LLM-phrased* questions.
  Real user logs (module 5) are the check on that bias.